# GWAS Catalog Associations

This notebook retrieves curated SNP-trait associations from the GWAS Catalog API for each gene identified in the previous Ensembl gene-region step.

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
gene_regions_file = Path("../data/interim/ensembl/gene_regions_grch38.csv")

out_dir = Path("../data/interim/gwas_catalog")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "associations.csv"

In [3]:
gene_regions = pd.read_csv(gene_regions_file)

gene_regions.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,GRCh38,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,GRCh38,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST


In [4]:
genes_for_gwas = gene_regions.loc[
    gene_regions["lookup_status"] == "found"
].copy()

genes_for_gwas = (
    genes_for_gwas
    .dropna(subset=["official_gene_symbol"])
    .drop_duplicates(subset=["official_gene_symbol"])
    .reset_index(drop=True)
)

print("Genes available for GWAS Catalog query:", len(genes_for_gwas))

genes_for_gwas.head()

Genes available for GWAS Catalog query: 51


,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,GRCh38,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,GRCh38,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST


In [5]:
BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

HEADERS = {
    "Accept": "application/json"
}

REQUEST_DELAY = 0.1

In [6]:
def gwas_api_request(endpoint, params=None, max_retries=3):
    """
    Send a GET request to the GWAS Catalog REST API.
    """
    url = f"{BASE_URL}{endpoint}"

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=HEADERS,
            timeout=60
        )

        if response.ok:
            return response.json()

        print("Request failed")
        print("URL:", response.url)
        print("Status code:", response.status_code)
        print("Response:", response.text[:500])

        if attempt < max_retries - 1:
            time.sleep(2)
        else:
            response.raise_for_status()

In [7]:
def fetch_paginated(endpoint, params=None, embedded_key=None, max_pages=None):
    """
    Fetch all pages from a paginated GWAS Catalog endpoint.
    """
    params = params.copy() if params else {}
    params.setdefault("size", 200)

    all_records = []
    page = 0
    total_pages = 1

    while page < total_pages:
        params["page"] = page

        data = gwas_api_request(endpoint, params=params)

        if "_embedded" not in data:
            break

        embedded = data["_embedded"]

        if embedded_key is None:
            embedded_key = list(embedded.keys())[0]

        records = embedded.get(embedded_key, [])
        all_records.extend(records)

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)

        page += 1
        time.sleep(REQUEST_DELAY)

        if max_pages is not None and page >= max_pages:
            break

    return all_records

In [8]:
def list_to_string(value):
    """
    Convert list values returned by the API into semicolon-separated strings.
    """
    if isinstance(value, list):
        return "; ".join(str(v) for v in value)
    return value

In [9]:
def extract_trait_names(efo_traits):
    """
    Extract trait names from the efo_traits field.
    """
    if not isinstance(efo_traits, list):
        return None

    traits = []

    for trait in efo_traits:
        if isinstance(trait, dict):
            name = (
                trait.get("efo_trait")
                or trait.get("trait")
                or trait.get("label")
            )

            if name:
                traits.append(name)

    return "; ".join(traits) if traits else None


def normalize_trait_id(value):
    """
    Normalize ontology identifiers so that MONDO:0005252,
    MONDO_0005252, and URI-style values can be compared.
    """
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    value = value.rsplit("/", 1)[-1]
    value = value.replace(":", "_")

    return value


def extract_trait_ids(efo_traits):
    """
    Extract ontology identifiers from the efo_traits field.
    """
    if not isinstance(efo_traits, list):
        return None

    ids = []

    for trait in efo_traits:
        if isinstance(trait, dict):
            trait_id = (
                trait.get("efo_id")
                or trait.get("short_form")
                or trait.get("uri")
                or trait.get("id")
            )

            trait_id = normalize_trait_id(trait_id)

            if trait_id:
                ids.append(trait_id)

    return "; ".join(ids) if ids else None

In [10]:
def extract_rsid_and_allele(snp_effect_allele):
    """
    Extract rsID and risk/effect allele from GWAS Catalog values such as:
    rs12345-A

    If the allele is not available, only the rsID is returned.
    """
    if not isinstance(snp_effect_allele, list) or len(snp_effect_allele) == 0:
        return None, None

    value = snp_effect_allele[0]

    if not isinstance(value, str):
        return None, None

    if "-" in value:
        rsid, allele = value.split("-", 1)
        return rsid, allele

    return value, None

In [11]:
def normalize_association(association, query_source=None, query_value=None):
    rsid, risk_allele = extract_rsid_and_allele(
        association.get("snp_effect_allele")
    )

    return {
        "query_source": query_source,
        "query_value": query_value,
        "association_id": association.get("association_id"),
        "rsID": rsid,
        "risk_allele": risk_allele,
        "p_value": association.get("p_value"),
        "risk_frequency": association.get("risk_frequency"),
        "mapped_genes": list_to_string(association.get("mapped_genes")),
        "trait_name": extract_trait_names(association.get("efo_traits")),
        "trait_id": extract_trait_ids(association.get("efo_traits")),
        "accession_id": association.get("accession_id"),
        "pubmed_id": association.get("pubmed_id"),
        "first_author": association.get("first_author"),
    }

In [12]:
all_associations = []

for _, gene_row in genes_for_gwas.iterrows():
    gene_symbol = gene_row["official_gene_symbol"]

    print("Querying GWAS Catalog for:", gene_symbol)

    gene_params = {
        "mapped_gene": gene_symbol,
        "extended_geneset": "false",
        "sort": "p_value",
        "direction": "asc",
        "size": 200
    }

    gene_associations_raw = fetch_paginated(
        endpoint="/v2/associations",
        params=gene_params,
        embedded_key="associations"
    )

    normalized_rows = [
        normalize_association(
            assoc,
            query_source="mapped_gene",
            query_value=gene_symbol
        )
        for assoc in gene_associations_raw
    ]

    all_associations.extend(normalized_rows)

print("Total raw association rows:", len(all_associations))

Querying GWAS Catalog for: APOE
Querying GWAS Catalog for: B2M
Querying GWAS Catalog for: C1QA
Querying GWAS Catalog for: C1QB
Querying GWAS Catalog for: C1QC
Querying GWAS Catalog for: C1R
Querying GWAS Catalog for: C2
Querying GWAS Catalog for: CAPG
Querying GWAS Catalog for: CCL2
Querying GWAS Catalog for: CCL3
Querying GWAS Catalog for: CCL7
Querying GWAS Catalog for: CD14
Querying GWAS Catalog for: CD163
Querying GWAS Catalog for: CFD
Querying GWAS Catalog for: CFH
Querying GWAS Catalog for: CSF1
Querying GWAS Catalog for: CSF1R
Querying GWAS Catalog for: CST3
Querying GWAS Catalog for: CTSA
Querying GWAS Catalog for: CTSC
Querying GWAS Catalog for: CTSL
Querying GWAS Catalog for: CTSZ
Querying GWAS Catalog for: CXCL1
Querying GWAS Catalog for: CXCL2
Querying GWAS Catalog for: CXCL3
Querying GWAS Catalog for: FN1
Querying GWAS Catalog for: GAA
Querying GWAS Catalog for: GUSB
Querying GWAS Catalog for: HEXB
Querying GWAS Catalog for: IFI30
Querying GWAS Catalog for: IL1RN
Querying 

In [13]:
associations_df = pd.DataFrame(all_associations)

associations_df.head()

,query_source,query_value,association_id,rsID,risk_allele,p_value,risk_frequency,mapped_genes,trait_name,trait_id,accession_id,pubmed_id,first_author
0,mapped_gene,APOE,101328355,rs7412,?,0.0,NR,APOE,low density lipoprotein cholesterol measurement,EFO_0004611,GCST90239655,34887591,Graham SE
1,mapped_gene,APOE,101331036,rs7412,T,0.0,0.0750304,APOE,low density lipoprotein cholesterol measurement,EFO_0004611,GCST90239658,34887591,Graham SE
2,mapped_gene,APOE,87309502,rs7412,T,0.0,NR,APOE,apolipoprotein B measurement,EFO_0004615,GCST90019496,33462484,Sinnott-Armstrong N
3,mapped_gene,APOE,101338523,rs7412,?,0.0,NR,APOE,non-high density lipoprotein cholesterol measu...,EFO_0005689,GCST90239667,34887591,Graham SE
4,mapped_gene,APOE,101343626,rs7412,?,0.0,NR,APOE,total cholesterol measurement,EFO_0004574,GCST90239673,34887591,Graham SE


In [14]:
if not associations_df.empty:
    associations_df = associations_df.drop_duplicates(
        subset=["query_value", "association_id"]
    )

    associations_df["p_value"] = pd.to_numeric(
        associations_df["p_value"],
        errors="coerce"
    )

    associations_df = associations_df.sort_values(
        ["query_value", "p_value"],
        ascending=[True, True],
        na_position="last"
    ).reset_index(drop=True)

associations_df.shape

(12319, 13)

In [15]:
selected_trait_ids = {
    "MONDO_0005252",  # Heart failure
    "MONDO_0005010",  # Coronary artery disease
    "MONDO_0005068",  # Myocardial infarction
}

selected_trait_terms = [
    "heart failure",
    "coronary artery disease",
    "myocardial infarction",
]

In [16]:
def is_selected_trait(row):
    trait_ids = row.get("trait_id")
    trait_name = row.get("trait_name")

    id_match = False
    name_match = False

    if pd.notna(trait_ids):
        ids = [
            normalize_trait_id(tid)
            for tid in str(trait_ids).split(";")
        ]

        id_match = any(tid in selected_trait_ids for tid in ids)

    if pd.notna(trait_name):
        trait_name_lower = str(trait_name).lower()

        name_match = any(
            term in trait_name_lower
            for term in selected_trait_terms
        )

    return id_match or name_match


if not associations_df.empty:
    associations_df["is_selected_trait"] = associations_df.apply(
        is_selected_trait,
        axis=1
    )
else:
    associations_df["is_selected_trait"] = pd.Series(dtype=bool)

associations_df.to_csv(out_csv, index=False)

associations_df["is_selected_trait"].value_counts(dropna=False)

is_selected_trait
False    12250
True        69
Name: count, dtype: int64

In [18]:
summary = {
    "genes_available_for_query": int(len(genes_for_gwas)),
    "genes_with_gwas_associations": int(
        associations_df["query_value"].nunique()
    ) if not associations_df.empty else 0,
    "associations": int(len(associations_df)),
    "selected_trait_associations": int(
        associations_df["is_selected_trait"].sum()
    ) if not associations_df.empty else 0,
    "unique_rsIDs_with_associations": int(
        associations_df["rsID"].nunique()
    ) if not associations_df.empty else 0,
    "assembly_for_gene_regions": "GRCh38",
}

summary

{'genes_available_for_query': 51,
 'genes_with_gwas_associations': 49,
 'associations': 12319,
 'selected_trait_associations': 69,
 'unique_rsIDs_with_associations': 1764,
 'assembly_for_gene_regions': 'GRCh38'}